# Quantize the own model

Exports run04's `best.pt` to ONNX (FP32) and quantizes it to INT8: post-training static
quantization through ONNX Runtime, calibrated on training images (`quantize_model.py`).
That's all this notebook does. **A CPU runtime is enough, and costs no compute units.**

- Accuracy: `compare_models.ipynb`, with the other models.
- Latency: `benchmark.py`, on the machine you care about.

## 1. Settings + mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"
RUN_NAME = "run04_long"
IMG_SIZE = 320

runDir = f"{RUN_ROOT}/{RUN_NAME}"
quantDir = f"{runDir}/quantized"

## 2. Download the dataset

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

## 3. Clone (or update) the repo

In [ ]:
import os
import sys

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt
!git -C {repoDirectory} log --oneline -1

sys.path.insert(0, repoDirectory)

## 4. Export + quantize

`export_model.py` checks that the ONNX model's outputs match PyTorch's. Both models are
saved to `runDir/quantized` on Drive, where `compare_models.ipynb` reads the INT8 one.

In [ ]:
import json

os.makedirs(quantDir, exist_ok=True)
manifestPath = f"{runDir}/manifest.json"
with open(manifestPath) as f:
    manifest = json.load(f)
with open(f"{runDir}/classes.json") as f:
    classNames = json.load(f)
numClasses = len(classNames)

MODELS = {"fp32": f"{quantDir}/model_fp32.onnx", "int8": f"{quantDir}/model_int8.onnx"}

!python {repoDirectory}/export_model.py --checkpoint {runDir}/best.pt --num-classes {numClasses} --img-size {IMG_SIZE} --out "{MODELS['fp32']}"
!python {repoDirectory}/quantize_model.py --onnx "{MODELS['fp32']}" --out "{MODELS['int8']}" --manifest {manifestPath} --img-size {IMG_SIZE}

for name, path in MODELS.items():
    print(f"{name:6s} {os.path.getsize(path) / 1e6:6.2f} MB")

## 5. Download the models, for `benchmark.py`

Then, in the repo, on your own machine:

```
python benchmark.py fp32=model_fp32.onnx int8=model_int8.onnx --threads 1 2 4
```

In [ ]:
from google.colab import files

for path in MODELS.values():
    files.download(path)